<a href="https://colab.research.google.com/github/sonyxoxo/ipd/blob/main/%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%B0_%D0%B2%D1%80%D0%B5%D0%BC%D0%B5%D0%BD%D0%B8_%D0%B8_%D0%BF%D0%BE%D0%BB%D0%BD%D0%BE%D1%82%D1%8B_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 2. Время и полнота измерений

Таблица считывается из репозитория **один раз**. Здесь оцениваются также минуты, для которых в файле вообще нет строки. Пропуски не восстанавливаются интерполяцией.

## Имена переменных

- `source` — файл в репозитории; `measurements` — все его измерения в `pandas`.
- `minutes` — полная минутная сетка от первой до последней даты.
- `output_dir` — папка результатов выбранного файла.
- `coverage_table` — полнота активной мощности по месяцам.
- `run_table` — непрерывные последовательности отсутствующих измерений.
- `expected_in_record` — число ожидаемых минут **внутри фактического периода записи**.
- `expected_calendar` — число минут целого календарного месяца, включая дни до начала или после конца файла.
- `valid_power` — число минут с пригодной активной мощностью.
- `coverage_record_pct` — доля пригодных минут в пределах периода записи, в процентах.
- `coverage_calendar_pct` — доля пригодных минут во всём календарном месяце, в процентах.
- `missing_or_invalid` — минуты с отсутствующей/непригодной мощностью.
- `crosses_month` — пересекает ли непрерывный пропуск границу месяца.

In [1]:
# Одна настройка пути к репозиторию. Исходные файлы не загружаются вручную.
from pathlib import Path
import subprocess
import sys

repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipd_analysis.py").is_file()), None)
if repo_dir is None:
    repo_dir = Path("/content/ipd")
    if not (repo_dir / "ipd_analysis.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sonyxoxo/ipd.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from ipd_analysis import choose_source, load_measurements, regular_minutes, result_folder, save_csv, save_run_metadata

# Если в data/raw несколько файлов, задайте относительный путь явно:
# source = choose_source("data/raw/household_power_consumption.zip")
source = choose_source()
measurements = load_measurements(source)  # Один pd.read_csv на весь ноутбук.
output_dir = result_folder(source)
output_dir.mkdir(parents=True, exist_ok=True)
print("Источник:", source)
print("Строк:", len(measurements))

Источник: /content/ipd/data/raw/household_power_consumption.zip
Строк: 2075259


## Расчёт полноты и последовательностей пропусков

Сначала `regular_minutes()` добавляет в **расчётную** таблицу отсутствующие временные метки. Обратите внимание: добавляются только строки со значениями `NaN`, не измеренные значения мощности.

`monthly_coverage()` не предполагает, что каждый файл содержит полный месяц. `missing_runs()` ищет непрерывные пропуски на единой временной сетке, поэтому пропуск между двумя соседними строками не теряется.

In [2]:
from ipd_analysis import monthly_coverage, missing_runs

minutes = regular_minutes(measurements)
coverage_table = monthly_coverage(minutes)
run_table = missing_runs(minutes)
display(coverage_table)
display(run_table.head(10))
save_csv(coverage_table, output_dir / "02_monthly_coverage.csv")
save_csv(run_table, output_dir / "02_missing_runs.csv")
print("Всего минут без корректной мощности:", int(coverage_table["missing_or_invalid"].sum()))

,month,expected_in_record,valid_power,expected_calendar,missing_or_invalid,coverage_record_pct,coverage_calendar_pct
0,2006-12,21996,21992,44640,4,99.981815,49.265233
1,2007-01,44640,44638,44640,2,99.995520,99.995520
2,2007-02,40320,40318,40320,2,99.995040,99.995040
3,2007-03,44640,44639,44640,1,99.997760,99.997760
4,2007-04,43200,39477,43200,3723,91.381944,91.381944
5,2007-05,44640,44640,44640,0,100.000000,100.000000
6,2007-06,43200,43157,43200,43,99.900463,99.900463
7,2007-07,44640,44509,44640,131,99.706541,99.706541
8,2007-08,44640,44618,44640,22,99.950717,99.950717
9,2007-09,43200,43198,43200,2,99.995370,99.995370


,start,end,minutes,crosses_month
0,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,False
1,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,False
2,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,False
3,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,False
4,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,False
5,2010-03-20 03:52:00,2010-03-21 13:38:00,2027,False
6,2009-08-13 05:00:00,2009-08-13 19:50:00,891,False
7,2007-07-15 16:49:00,2007-07-15 18:11:00,83,False
8,2008-12-10 10:48:00,2008-12-10 11:57:00,70,False
9,2007-07-15 18:21:00,2007-07-15 19:07:00,47,False


Всего минут без корректной мощности: 25979


## Воспроизводимость результата

Последняя ячейка сохраняет `metadata.json` с SHA-256 исходного файла, хешами исполняемого кода, версиями библиотек и датой запуска. Поэтому любой результат можно связать с конкретным источником и версией обработки.

In [3]:
save_run_metadata(source, output_dir, {"notebook": 'проверка_времени_и_полноты_данных.ipynb'})

PosixPath('/content/ipd/результаты/household_power_consumption-b916e64b/metadata.json')